# Asshifa AI Model Training (Google Colab)

Trains a small LLM on *Ash-Shifa* by Qadi Iyad (rahimahullah) using Unsloth.

**Pipeline:** Continued pretraining on Shifa corpus → Adab instruction tuning → GGUF export for Ollama.

---

## Step 1 — Check GPU

In [ ]:
import torch, os, sys
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Mount Google Drive (optional but recommended)

Saves checkpoints across sessions so you don't lose progress if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/asshifa-training"
!mkdir -p {DRIVE_DIR}

## Step 3 — Install Dependencies

In [ ]:
print("Installing Unsloth and dependencies...")
!pip install -q pip --upgrade
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade --no-deps --force-reinstall -q unsloth
!pip install -q transformers datasets trl accelerate bitsandbytes
print("\nAll packages installed.")

## Step 4 — Clone Repo & Prepare Corpus

In [ ]:
import os, sys

REPO_URL = "https://github.com/sahilhasnain/islamic-knowledge"
if not os.path.exists("/content/islamic-knowledge"):
    !git clone --branch ai-model --depth 1 {REPO_URL} /content/islamic-knowledge

%cd /content/islamic-knowledge

# Generate corpus by running the script directly
!python "ai-model/prepare_corpus.py"

print("\nCorpus ready.")

## Step 5 — Model Selection

Choose a model. On the free T4 (16GB VRAM):
- **3B** (~5-6 GB VRAM) — fast, safe, plenty of room
- **7B** (~10-11 GB VRAM) — more capable, fits on T4 with QLoRA

Run the cell below for 7B. If you hit OOM, restart runtime and use 3B instead.

In [ ]:
# Pick your model size:
MODEL_SIZE = "7b"  # change to "3b" if T4 runs out of memory
print(f"Using model: {MODEL_SIZE}")

## Step 6 — Run Training

This cell runs both phases (continued pretraining + adab tuning) and exports GGUF.

**Expected time on T4:**
- 3B model: ~15-20 min per epoch (corpus) + ~5 min (adab)
- 7B model: ~30-40 min per epoch (corpus) + ~10 min (adab)

3 epochs on corpus + 2 epochs on adab = roughly 1-2 hours total.

In [ ]:
import json, sys, os
from pathlib import Path

# Constants (duplicated from train.py to avoid import issues with hyphen in dir name)
MODELS = {
    "3b": {"name": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", "max_seq": 8192},
    "7b": {"name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "max_seq": 8192},
    "8b": {"name": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", "max_seq": 8192},
}
CORPUS_PATH = Path("ai-model/data/corpus.txt")
ADAB_PATH = Path("ai-model/data/adab-examples.jsonl")
OUTPUT_DIR = Path("ai-model/output")

model_cfg = MODELS[MODEL_SIZE]
print(f"Model: {model_cfg['name']}")
print(f"Max seq length: {model_cfg['max_seq']}")
print(f"Corpus: {CORPUS_PATH}")
print(f"Adab: {ADAB_PATH}")

# ----------------------------------------------------------------------
# Phase 1: Continued pretraining
# ----------------------------------------------------------------------
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_cfg["name"],
    max_seq_length=model_cfg["max_seq"],
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5" if "Qwen" in model_cfg["name"] else "llama-3"
)

# Add Arabic honorific tokens
arabic_tokens = [
    "\ufdfa", "عليه السلام", "عز وجل", "صلى الله عليه وسلم",
    "رضي الله عنه", "رضي الله عنها", "رحمة الله تعالى عليه",
    "سبحانه وتعالى"
]
added = 0
for t in arabic_tokens:
    if t not in tokenizer.get_vocab():
        tokenizer.add_tokens([t])
        added += 1
if added:
    model.resize_token_embeddings(len(tokenizer))
    print(f"Added {added} Arabic honorific tokens")

# LoRA setup
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Read & chunk corpus
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

chunk_size = model_cfg["max_seq"] * 3
paragraphs = raw_text.split("\n\n")
chunks, current, current_len = [], [], 0
for para in paragraphs:
    para = para.strip()
    if not para:
        continue
    if current_len + len(para) > chunk_size and current:
        chunks.append("\n\n".join(current))
        current, current_len = [], 0
    current.append(para)
    current_len += len(para)
if current:
    chunks.append("\n\n".join(current))

corpus_dataset = Dataset.from_list([{"text": c} for c in chunks])
print(f"Corpus: {len(chunks)} chunks, {len(raw_text):,} chars")

corpus_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/asshifa-training/phase1",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    num_train_epochs=3.0,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    ddp_find_unused_parameters=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
)

trainer_pt = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=corpus_args,
    train_dataset=corpus_dataset,
    dataset_text_field="text",
    max_seq_length=model_cfg["max_seq"],
    dataset_num_proc=2,
    packing=True,
)

print("=" * 60)
print("Phase 1: Continued pretraining on Shifa Shareef corpus")
print("=" * 60)
trainer_pt.train()

# ----------------------------------------------------------------------
# Phase 2: Adab instruction tuning
# ----------------------------------------------------------------------
if ADAB_PATH.exists():
    with open(ADAB_PATH, "r") as f:
        adab_data = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(adab_data)} adab examples")

    def format_adab(example):
        return {"conversations": [
            {"from": "human", "value": example["instruction"]},
            {"from": "gpt", "value": example["response"]},
        ]}

    adab_dataset = Dataset.from_list([format_adab(d) for d in adab_data])
    adab_dataset = adab_dataset.map(
        lambda x: {"text": tokenizer.apply_chat_template(
            x["conversations"], tokenize=False)}
    )

    adab_args = TrainingArguments(
        output_dir="/content/drive/MyDrive/asshifa-training/phase2",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2.0,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        save_strategy="epoch",
        report_to="none",
        ddp_find_unused_parameters=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
    )

    trainer_sft = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=adab_args,
        train_dataset=adab_dataset,
        dataset_text_field="text",
        max_seq_length=model_cfg["max_seq"],
        dataset_num_proc=2,
        packing=False,
    )

    # Mask user prompts during training
    if "llama-3" in model_cfg["name"]:
        trainer_sft = train_on_responses_only(
            trainer_sft,
            instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
            response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
        )
    elif "qwen" in model_cfg["name"]:
        trainer_sft = train_on_responses_only(
            trainer_sft,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )

    print("=" * 60)
    print("Phase 2: Adab instruction tuning")
    print("=" * 60)
    trainer_sft.train()
else:
    print(f"WARNING: {ADAB_PATH} not found, skipping phase 2")

# ----------------------------------------------------------------------
# Step 7: Save & Export
# ----------------------------------------------------------------------
OUTPUT_DIR = Path("/content/drive/MyDrive/asshifa-training/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(OUTPUT_DIR / "lora"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "lora"))
print(f"LoRA saved to {OUTPUT_DIR / 'lora'}")

print("Exporting GGUF (may take 10-15 min)...")
model.save_pretrained_gguf(
    str(OUTPUT_DIR / "gguf"),
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF saved to {OUTPUT_DIR / 'gguf'}")

# Also copy to Colab instance for easy download
import shutil
shutil.copytree(str(OUTPUT_DIR / "gguf"), "/content/asshifa-gguf", dirs_exist_ok=True)
!zip -r /content/asshifa-gguf.zip /content/asshifa-gguf

print("\n" + "=" * 60)
print("Training complete!")
print("=" * 60)
print("1. Download asshifa-gguf.zip from Files tab (left sidebar)")
print("2. On your machine:")
print("   unzip asshifa-gguf.zip")
print("   ollama create asshifa -f asshifa-gguf/Modelfile")
print("   ollama run asshifa")

## Troubleshooting

### Out of Memory (OOM)
- Restart runtime: **Runtime -> Factory reset runtime**
- Change `MODEL_SIZE` in Step 5 to `"3b"`

### Session Disconnects
- Checkpoints and final model saved to Google Drive (`asshifa-training/`)
- Resume from latest checkpoint: pass `resume_from_checkpoint=True` to `.train()`

### Slow Training
- T4: ~7 TFLOPS fp16. Full run ~1-2 hours.
- Enable Background execution in Colab settings.